# Lesson 6: A three-tier router, and how to judge whether it's worth it

*Module 3 · about 15 minutes · API key required*

In Lesson 1 we saw a gap of about 107× between the cheapest and most expensive models. Routing is how you use that gap: send each request to the cheapest model that can handle it, and only pay for a bigger model when you need one. Of all the levers in this course, it has the most potential and it's the easiest to get subtly wrong.

We'll build a simple **cascade**. Every request starts on the cheapest model, we check the answer, and we escalate to a bigger model only when the check says to. Then we'll ask the question that decides whether the router is actually a saving. Not "is it cheaper per call?" but "is it cheaper per *correct* answer, once you count what wrong answers cost to clean up?"

By the end you should be able to:

1. Run the same workload with everything on one model and through a cascade, and compare cost and accuracy.
2. Explain which signals a cascade can use to decide when to escalate, and why some are more reliable than others.
3. Compute cost per solved task including cleanup, and find the point where the decision flips.


### Routers and cascades

There are two main designs:

- A **router** looks at the request *before* any model runs and picks one. It might use rules ("anything mentioning a refund over \$500 goes to the big model"), a small classifier, or a cheap model asked to rate difficulty.
- A **cascade** *tries* the cheap model first, checks the result, and escalates if the check fails. You sometimes pay twice (cheap attempt, then the expensive one), so it only pays off if most requests are settled by the first, cheap attempt.

Either way, the design depends on the **escalation signal**: how you decide an answer isn't good enough. From most to least reliable, the options are roughly:

1. **Deterministic checks.** Does the output parse, match the required format, pass a validation rule? These are cheap and hard to fool.
2. **Calibrated confidence.** A score trained on your own labelled data to predict correctness. The 2026 UCCI paper (arXiv 2605.18796) used this to cut cost by 31% (95% CI 27–35%) on 75,000 real production queries at unchanged accuracy.
3. **Asking the model how sure it is.** Easy to build, but models are often confidently wrong, so treat it as a weak signal.

We'll combine 1 and 3. Tune the settings on a validation set, never by guessing in production.

The tiers come from `MODEL_FLOOR`, `MODEL_MID`, and `MODEL_FRONTIER` in your `.env`. If your key can only reach one model, set all three to it. The cost-per-solved-task part of the lesson still works.


### How these notebooks work

Run the cells in order, top to bottom. Before each code cell there's a short explanation of what it does and what to look at in the output. After the important ones there's a note on how to read what you got. Your numbers won't match mine exactly, because models are non-deterministic and prices change, so the notes describe what to look for rather than quoting fixed values.

A few conventions:

- **In class:** notes are cues for when we run this together. If you're working alone, just read them as a prompt to stop and think.
- Every notebook that spends money ends with a **ledger**: one row per API call and the total you spent.
- The **Check yourself** questions at the end have answers hidden under a click. Try them before you look.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, vendor_tokens, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
LIVE = cfg.live


  Provider : anthropic
  floor    : claude-haiku-4-5
  mid      : claude-sonnet-5
  frontier : claude-opus-5
  Cache    : explicit cache_control; read/write are separate buckets.
Switch with LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env
  Rate card: verified 5 Sep 2026 — re-check before presenting.


That cell reads your `.env`, picks OpenAI or Anthropic depending on which key it finds, and prints the three model tiers the notebook will use (floor, mid, frontier).

If the banner names a provider, the live cells will make real calls. Every lesson costs cents, not dollars. If it says `offline`, all the arithmetic still runs, but cells that need a model's answer print a placeholder and tell you they can't draw a conclusion. You can read an offline run, but it's no substitute for a live one in the caching, compression, and routing lessons.

To switch vendors, set `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and run the cell again.


In [2]:
print("Model ladder for this key:")
print(f"  floor     {MODELS.floor}")
print(f"  mid       {MODELS.mid}")
print(f"  frontier  {MODELS.frontier}")


Model ladder for this key:
  floor     claude-haiku-4-5
  mid       claude-sonnet-5
  frontier  claude-opus-5


---
## 1. A mixed workload

Twelve tasks, roughly the mix many production workloads show: most are easy and a few are hard.

- **8 simple:** pull an order ID out of a sentence.
- **3 medium:** apply a refund policy to a customer's situation and answer YES or NO. The answers are deliberately mixed (one YES, two NO, for different reasons), so a model can't score well by always saying YES.
- **1 hard:** apply the same policy to three shipments at once and list which ones qualify.

The "about 70% easy" split is a planning assumption that shows up often in routing research, not a law. Routing only pays when most of your traffic really is easy. If 80% of your requests are hard, a cascade will cost *more*, because you pay for a cheap attempt that fails before paying for the real one.


In [3]:
import re

TASKS = (
    [dict(q=f'Extract the order id from: "Order NW-{1000 + i} was delayed 3 days." Reply with the id only.',
          gold=f"NW-{1000 + i}", klass="simple", trust_format=True) for i in range(8)]
    + [dict(q=("Policy: refunds under $500 auto-approve if the delay is over 48h and the customer has "
               f"fewer than 3 prior claims. Customer: ${amt} claim, {delay}h delay, {prior} prior claims. "
               "Auto-approve? Answer YES or NO first."),
            gold=gold, klass="medium", trust_format=False)
       for amt, delay, prior, gold in [(300, 60, 0, "YES"), (520, 60, 1, "NO"), (450, 60, 3, "NO")]]
    + [dict(q=("Three shipments: A delayed 50h, claim $480, 2 prior claims; B delayed 20h, claim $100, "
               "0 prior; C delayed 72h, claim $900, 1 prior. Policy: auto-approve if delay > 48h AND "
               "claim < $500 AND prior < 3; otherwise escalate. Which auto-approve? "
               "Reply with the letters only."),
            gold="A", klass="hard", trust_format=False)]
)

def graded(task, answer):
    """Strict grading per task type, so a vague answer can't pass by accident."""
    a = answer.strip()
    if task["klass"] == "simple":
        return re.search(rf"\b{re.escape(task['gold'])}\b", a) is not None
    if task["klass"] == "medium":
        first = re.match(r"\W*(YES|NO)\b", a, re.I)
        return bool(first) and first.group(1).upper() == task["gold"]
    first_line = a.splitlines()[0] if a else ""
    return set(re.findall(r"\b([ABC])\b", first_line)) == set(task["gold"])

print(f"{len(TASKS)} tasks:", pd.Series([t["klass"] for t in TASKS]).value_counts().to_dict())


12 tasks: {'simple': 8, 'medium': 3, 'hard': 1}


---
## 2. The baseline: everything on the mid-tier model

This is what many teams ship: take the model that worked well in the prototype and use it for everything. We run all 12 tasks on it and record the cost and the accuracy. Everything after this is compared with these two numbers.


In [4]:
baseline = []
for t in TASKS:
    r = complete(t["q"], model=MODELS.mid, max_tokens=80, label=f"baseline {t['klass']}")
    baseline.append(dict(klass=t["klass"], cost=r.usd,
                         ok=None if r.fallback else graded(t, r.text), ans=r.text[:50]))

bdf = pd.DataFrame(baseline)
LIVE_RUN = bdf.ok.notna().all()
B_COST = bdf.cost.sum()
B_ACC = bdf.ok.mean() if LIVE_RUN else None
print(f"\nBASELINE  cost={usd(B_COST)}  "
      + (f"accuracy={B_ACC:.0%}" if LIVE_RUN else "accuracy=n/a (no live answers)")
      + f"  ({len(TASKS)} tasks, all on {MODELS.mid})")


baseline simple                               $0.000150   in=40      out=7      cw=0       cr=0       


baseline simple                               $0.000150   in=40      out=7      cw=0       cr=0       


baseline simple                               $0.000150   in=40      out=7      cw=0       cr=0       


baseline simple                               $0.000150   in=40      out=7      cw=0       cr=0       


baseline simple                               $0.000150   in=40      out=7      cw=0       cr=0       


...

baseline simple                               $0.000150   in=40      out=7      cw=0       cr=0       


baseline medium                               $0.000978   in=89      out=80     cw=0       cr=0       


baseline medium                               $0.000978   in=89      out=80     cw=0       cr=0       


baseline medium                               $0.000978   in=89      out=80     cw=0       cr=0       


baseline hard                                 $0.000286   in=128     out=3      cw=0       cr=0       

BASELINE  cost=$0.004420  accuracy=100%  (12 tasks, all on claude-sonnet-5)


---
## 3. The cascade

Every task starts on the floor model. After each answer we apply up to two checks:

1. **Format check (deterministic).** Is the answer in the shape the task asked for: an order ID, a YES/NO, a list of letters? If not, escalate straight away. This costs nothing.
2. **Confidence check (the model's own opinion).** We ask the floor model whether the answer is fully determined by the question: HIGH, MEDIUM, or LOW. Anything below HIGH escalates. This costs an extra call.

We don't run the second check on everything. For the extraction tasks, an answer in the right format is almost certainly right, because the model only has to copy the ID from the question, so the free format check is enough (`trust_format=True`). For the policy questions, a well-formed YES says nothing about whether the reasoning behind it was right, so those get the probe as well. Use the cheapest check that's reliable enough for each kind of task. A probe on every request can easily cost more than the answers it's checking.

If the mid-tier answer still fails, we go to the frontier model. The ledger lines show each call, including the probe calls. They're part of the real cost of a cascade and easy to forget.

Look at the `path` for each task: most of the simple ones should stop at `floor`.

> **In class:** before running the reframe in section 4, ask the room what they'd report if the cascade came out cheaper. Then show why that number isn't the whole story.


In [5]:
def format_ok(task, answer):
    a = answer.strip()
    if task["klass"] == "simple":
        return re.fullmatch(r"\W*NW-\d{4}\W*", a) is not None
    if task["klass"] == "medium":
        return re.match(r"\W*(YES|NO)\b", a, re.I) is not None
    return re.fullmatch(r"[\sABC,and.]+", a.splitlines()[0] if a else "") is not None

def confidence_probe(task, answer):
    probe = (f"Question: {task['q']}\n\nProposed answer: {answer}\n\n"
             "Is this answer fully determined by the question, with no ambiguity? "
             "Reply with only HIGH, MEDIUM or LOW.")
    r = complete(probe, model=MODELS.floor, max_tokens=8, label="  probe")
    m = re.search(r"\b(HIGH|MEDIUM|LOW)\b", r.text.upper())
    return (m.group(1) if m else "UNCLEAR"), r.usd

def attempt(task, model, tier):
    r = complete(task["q"], model=model, max_tokens=80, label=f"{tier} {task['klass']}")
    if r.fallback:
        return r, r.usd, False, "offline"
    if not format_ok(task, r.text):
        return r, r.usd, False, "bad format"
    if task["trust_format"]:
        return r, r.usd, True, "format ok"
    level, probe_cost = confidence_probe(task, r.text)
    return r, r.usd + probe_cost, level == "HIGH", f"confidence {level}"

routed = []
for t in TASKS:
    spend, path, reasons = 0.0, [], []
    for tier, model in [("floor", MODELS.floor), ("mid", MODELS.mid), ("frontier", MODELS.frontier)]:
        r, c, accepted, why = attempt(t, model, tier)
        spend += c
        path.append(tier)
        reasons.append(why)
        if accepted or tier == "frontier" or r.fallback:
            break
    ok = None if r.fallback else graded(t, r.text)
    routed.append(dict(klass=t["klass"], path=">".join(path), cost=spend, ok=ok))
    print(f"   {t['klass']:<7} {'>'.join(path):<20} {usd(spend):>10}  "
          f"{'n/a' if ok is None else ('OK' if ok else 'WRONG')}   [{'; '.join(reasons)}]")

rdf = pd.DataFrame(routed)
R_COST = rdf.cost.sum()
R_ACC = rdf.ok.mean() if LIVE_RUN and rdf.ok.notna().all() else None


floor simple                                  $0.000074   in=34      out=8      cw=0       cr=0       
   simple  floor                 $0.000074  OK   [format ok]


floor simple                                  $0.000074   in=34      out=8      cw=0       cr=0       
   simple  floor                 $0.000074  OK   [format ok]


floor simple                                  $0.000074   in=34      out=8      cw=0       cr=0       
   simple  floor                 $0.000074  OK   [format ok]


floor simple                                  $0.000074   in=34      out=8      cw=0       cr=0       
   simple  floor                 $0.000074  OK   [format ok]


floor simple                                  $0.000074   in=34      out=8      cw=0       cr=0       
   simple  floor                 $0.000074  OK   [format ok]


...

floor hard                                    $0.000495   in=95      out=80     cw=0       cr=0       


mid hard                                      $0.000286   in=128     out=3      cw=0       cr=0       


  probe                                       $0.000169   in=129     out=8      cw=0       cr=0       


frontier hard                                 $0.000715   in=128     out=3      cw=0       cr=0       


  probe                                       $0.000169   in=129     out=8      cw=0       cr=0       
   hard    floor>mid>frontier    $0.001834  OK   [bad format; confidence MEDIUM; confidence LOW]


---
## 4. The results, per call

First the routing mix (which path each class of task took), then the straightforward comparison: total cost, accuracy, and cost per task.

If the cascade came out *more* expensive, that's a real result, not a bug. Two things decide whether a cascade can win. The first is the **price gap between tiers**. On our rate card Anthropic's floor model is about 2× cheaper than its mid model, while OpenAI's is about 10× cheaper. The narrower the gap, the less room there is to pay for failed cheap attempts and probe calls. The second is the **overhead of the checks**. A probe prompt includes the whole question plus the answer, so on tiny tasks it can cost more than the answer did.

Also look at the hard task's path. Sometimes the floor model's probe rates a correct answer from a bigger model as LOW, which is the weakness of self-assessment showing up live.

Published routing papers often report savings of 85–98%. Those come from benchmarks with a very wide gap between the cheap and expensive model and tidy, labelled traffic. RouteLLM, for example, kept 95% of GPT-4's quality on MT-Bench while sending only 14–26% of queries to the strong model. On a real production workload, UCCI measured 31%. Plan with the second kind of number.


In [6]:
print("ROUTING MIX")
show(rdf.groupby(["klass", "path"]).agg(n=("cost", "size"), cost=("cost", "sum"), accuracy=("ok", "mean")))

n = len(TASKS)
acc = lambda x: "n/a" if x is None else f"{x:.0%}"
print(f"\n{'':<22}{'total cost':>12}{'accuracy':>10}{'cost/task':>12}")
print(f"{'baseline (all mid)':<22}{usd(B_COST):>12}{acc(B_ACC):>10}{usd(B_COST / n):>12}")
print(f"{'3-tier cascade':<22}{usd(R_COST):>12}{acc(R_ACC):>10}{usd(R_COST / n):>12}")
print(f"\nCost difference per call: {1 - R_COST / B_COST:+.0%} "
      f"({'cascade cheaper' if R_COST < B_COST else 'cascade MORE expensive'})")


ROUTING MIX


,,n,cost,accuracy
klass,path,,,
hard,floor>mid>frontier,1,0.001834,1.0
medium,floor,3,0.001544,1.0
simple,floor,8,0.000592,1.0



                        total cost  accuracy   cost/task
baseline (all mid)       $0.004420      100%   $0.000368
3-tier cascade           $0.003970      100%   $0.000331

Cost difference per call: +10% (cascade cheaper)


### Now count what wrong answers cost

A per-call comparison assumes every answer is equally useful. It isn't. A wrong answer either has to be redone or, worse, it gets through to a customer or a business process and someone has to clean it up. So the number that matters is the **expected cost per solved task**:

$$E[\text{cost per solved task}] = \frac{C_{\text{attempt}}}{p_{\text{success}}} + L \times K_{\text{cleanup}}$$

- $C_{\text{attempt}}$ is the average cost of one attempt (what we measured above).
- Dividing by $p_{\text{success}}$ (the accuracy) accounts for retries: if only 80% of attempts succeed, you pay for 1.25 attempts per success.
- $L$ is the share of attempts whose wrong answer *leaks* into the real world. Here we make the pessimistic assumption that every wrong answer leaks, so $L = 1 - p_{\text{success}}$.
- $K_{\text{cleanup}}$ is what it costs a person to find and fix one leaked mistake. That's the refund issued in error, the support ticket, the apology.

Cleanup is usually measured in dollars per incident. API calls are measured in fractions of a cent. So even a small difference in accuracy can outweigh a large difference in API cost.

> **In class:** set `CLEANUP_COST` together. A support team, a hospital, and a bank will pick very different numbers, and that's the point.


In [7]:
CLEANUP_COST = 12.00   # dollars of staff time to find and fix one wrong answer. Pick an honest number.

def per_solved(total_cost, accuracy, n, cleanup):
    attempt_cost = total_cost / n
    return attempt_cost / max(accuracy, 1e-9) + (1 - accuracy) * cleanup

if B_ACC is None or R_ACC is None:
    print("No live accuracy numbers, so cost per solved task can't be computed. "
          "Run this notebook with an API key to see the comparison.")
else:
    b = per_solved(B_COST, B_ACC, n, CLEANUP_COST)
    r = per_solved(R_COST, R_ACC, n, CLEANUP_COST)
    print(f"{'':<22}{'cost per call':>15}{'cost per solved task':>23}")
    print(f"{'baseline':<22}{usd(B_COST / n):>15}{usd(b):>23}")
    print(f"{'cascade':<22}{usd(R_COST / n):>15}{usd(r):>23}")
    print()
    cheaper_call, cheaper_task = R_COST < B_COST, r < b
    if cheaper_call and cheaper_task:
        print("The cascade is cheaper per call AND per solved task.")
    elif cheaper_call and not cheaper_task:
        print("The cascade is cheaper per call but MORE expensive per solved task: "
              "the accuracy it gives up costs more in cleanup than it saves in API spend.")
    elif not cheaper_call and cheaper_task:
        print("The cascade costs more per call but less per solved task, because it gets more answers right.")
    else:
        print("The cascade is more expensive on both measures for this workload.")


                        cost per call   cost per solved task
baseline                    $0.000368              $0.000368
cascade                     $0.000331              $0.000331

The cascade is cheaper per call AND per solved task.


---
## 5. Where does the decision flip?

The measured result above is one data point: twelve tasks, one run. In practice you want to know how *sensitive* the decision is. How much worse could the cascade's accuracy get before it stops paying? And how does that depend on the cost of a mistake?

The table below keeps the API costs we measured and varies two things. Down the side is the cleanup cost per wrong answer. Across the top is how many percentage points *less* accurate the cascade is than the baseline (0 means equally accurate). Compare with the accuracy gap you measured in section 4. Each cell says which option has the lower cost per solved task.


In [8]:
if B_ACC is None:
    print("Needs live accuracy numbers.")
else:
    gaps = [0, 2, 5, 10]                           # cascade accuracy is this many points below baseline
    grid = {}
    for cu in [0, 1, 5, 12, 50]:
        row = {}
        for g in gaps:
            r_acc = max(B_ACC - g / 100, 0.01)
            b_cost = per_solved(B_COST, B_ACC, n, cu)
            r_cost = per_solved(R_COST, r_acc, n, cu)
            row[f"-{g} pts"] = "cascade" if r_cost < b_cost else "baseline"
        grid[f"${cu} cleanup"] = row
    show(pd.DataFrame(grid).T)


,-0 pts,-2 pts,-5 pts,-10 pts
$0 cleanup,cascade,cascade,cascade,cascade
$1 cleanup,cascade,baseline,baseline,baseline
$5 cleanup,cascade,baseline,baseline,baseline
$12 cleanup,cascade,baseline,baseline,baseline
$50 cleanup,cascade,baseline,baseline,baseline


**Reading the table.** If the cascade was more expensive per call in your run, the table says "baseline" everywhere, which is the right answer: a router that costs more and is no more accurate never pays. If the cascade was cheaper per call, look at the top row. When mistakes are free to fix, the cascade wins even if it's quite a bit less accurate, because only the API cost matters. As cleanup cost rises, even a two-point drop in accuracy wipes out the API saving. That's why the table, not the headline saving, is what belongs in a decision meeting: it shows under which conditions routing pays off.

It's also why the quality check should ship **in the same change as the router**. If you can't measure accuracy per route, you can't tell which row of this table you're in.


In [9]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.008390


,label,model,input,output,cache_write,cache_read,usd,note
0,baseline simple,claude-sonnet-5,40,7,0,0,0.000150,
1,baseline simple,claude-sonnet-5,40,7,0,0,0.000150,
2,baseline simple,claude-sonnet-5,40,7,0,0,0.000150,
3,baseline simple,claude-sonnet-5,40,7,0,0,0.000150,
4,baseline simple,claude-sonnet-5,40,7,0,0,0.000150,
5,baseline simple,claude-sonnet-5,40,7,0,0,0.000150,
6,baseline simple,claude-sonnet-5,40,7,0,0,0.000150,
7,baseline simple,claude-sonnet-5,40,7,0,0,0.000150,
8,baseline medium,claude-sonnet-5,89,80,0,0,0.000978,
9,baseline medium,claude-sonnet-5,89,80,0,0,0.000978,


---
## What to take away

- Routing uses the big price gap between models, but it only pays when most traffic can be handled by the cheap tier.
- The escalation signal matters more than the routing code. Prefer deterministic checks and calibrated scores over asking the model how confident it is.
- Count the probe calls and failed cheap attempts. They're part of the cascade's real cost.
- Judge a router on cost per *solved* task, including what wrong answers cost to clean up, not on cost per call.
- Expect savings nearer 30% on real traffic than the 85–98% in benchmark papers.


### Check yourself

**1. A cascade costs \$0.002 per task at 94% accuracy. The baseline costs \$0.006 per task at 97% accuracy. Cleanup costs \$20 per wrong answer. Which is cheaper per solved task?**

<details><summary>Show answer</summary>

Cascade: 0.002/0.94 + 0.06 × 20 ≈ \$0.0021 + \$1.20 = **\$1.20**. Baseline: 0.006/0.97 + 0.03 × 20 ≈ \$0.0062 + \$0.60 = **\$0.61**. The baseline wins easily: the three accuracy points are worth far more than the API saving.

</details>

**2. Why is asking the cheap model 'how confident are you?' a weak escalation signal?**

<details><summary>Show answer</summary>

Models are often confidently wrong, and a small model judging its own answer tends to be poorly calibrated: it says HIGH on answers that are wrong. A format check, a validation rule, or a confidence score calibrated on your own labelled data is much more reliable.

</details>

**3. Your traffic turns out to be 60% hard. What happens to a cheap-first cascade?**

<details><summary>Show answer</summary>

Most requests fail on the cheap tier and escalate, so you pay for a cheap attempt, a probe, *and* the expensive call. The cascade can end up costing more than sending everything to the expensive model. For hard-heavy traffic, route up front (classify before calling) instead of cascading.

</details>


### Try it on your own work

Sample 100 real requests and label each simple, medium, or hard. Price them all on the mid-tier model against a 70/20/10 split. Then ask someone in operations what one wrong answer costs to fix, and use that number to redo the comparison per solved task.
